> de modifica chunk size! 

augumentare

[What is BM25 (Best Matching 25) Algorithm](https://www.geeksforgeeks.org/nlp/what-is-bm25-best-matching-25-algorithm/) vs Chunking

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2 merge cu text in română?


[RAG with Semantic Text Splitting for PDF Documents](https://medium.com/@srmsiva1613/rag-with-semantic-text-splitting-for-pdf-documents-511dc6f18784)

[Build a RAG Pipeline in Python That Actually Works](https://dev.to/klement_gunndu/build-a-rag-pipeline-in-python-that-actually-works-28dg)

```
pip install langchain-openai langchain-chroma langchain-community \
            langchain-text-splitters chromadb beautifulsoup4
```

Pattern1 : Document loading and chunking that preserve context

In [12]:
print("Ceva")

Ceva


In [19]:
from langchain_community.document_loaders import WebBaseLoader ,PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import bs4

loader = PyPDFDirectoryLoader("data/cursuri_TRA")

docs = loader.load()


loader = WebBaseLoader(
    web_paths=["https://en.wikipedia.org/wiki/System","https://en.wikipedia.org/wiki/Industrial_processes","https://www.intechopen.com/chapters/59920","https://ro.wikipedia.org/wiki/Teoria_sistemelor","https://lilianweng.github.io/posts/2023-06-23-agent/"],
    header_template={"User-Agent": "Mozilla/5.0"},
    bs_kwargs={
        "parse_only":bs4.SoupStrainer(
            class_ =("mw-content-ltr mw-parser-output","mw-heading mw-heading3","mw-heading mw-heading2","mw-page-title-main","div","main","h1","h2","h3","a","p","post-title","post-header","post-content")
        )
    },
)

docs.extend(loader.load())

print(f"Loaded {len(docs)} page/documents")
pages = [page.page_content for page in loader.lazy_load()]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
    add_start_index =True,
)
splits = text_splitter.split_documents(docs)

print(f"Loaded {len(docs)} documnets, split into {len(splits)} chunks")

Loaded 380 page/documents
Loaded 380 documnets, split into 923 chunks


In [14]:
type(docs)

list

In [15]:
docs

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/System'}, page_content='SystemInterrelated entities that form a whole\nFor other uses, see System (disambiguation). For the set of rules that govern structure or behavior of people, see Social system. For the academic field, see Systems science. For the engineering, see Systems engineering.\n\nSystems can be isolated, closed, or open.\nA system is a group of interacting or interrelated elements that act according to a set of rules or set of constraints to form a unified whole.[1] A system, surrounded and influenced by its environment, is described by its boundaries, structure and purpose and is expressed in its functioning. Systems are the subjects of study of systems theory and other systems sciences.\nSystems have several common properties and characteristics, including structure, function(s), behavior and interconnectivity.\nEtymology[edit]\nThe term system comes from the Latin word systēma, in turn from Greek σύστημα syst

Pattern 2: Embeddings and vextor store setup

In [ ]:
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

embeddings = HuggingFaceEmbeddings(model_name= "BlackKakapo/stsb-xlm-r-multilingual-ro")

text_splitter = SemanticChunker(embeddings)
chunks = text_splitter.create_documents(pages)

texts = [chunk.page_content for chunk in chunks]

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db",
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs = {"k":4},
)


results = retriever.invoke("What is task decomposition?")
for doc in results:
    print(f"[Score chunks from index{doc.metadata.get("start_index",'?')}]")
    print(doc.page_content[:200])
    print("---")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

TypeError: TextEncodeInput must be Union[TextInputSequence, Tuple[InputSequence, InputSequence]]

In [26]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

emb = HuggingFaceEmbeddings(model_name= "BlackKakapo/stsb-xlm-r-multilingual-ro")

# sentences = ["This is an example sentence", "Each sentence is converted"]
text_splitter = SemanticChunker(emb)
chunks = text_splitter.create_documents(pages)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [35]:
from langchain_core.vectorstores import InMemoryVectorStore

texts = [chunk.page_content for chunk in chunks]

vectorstore = InMemoryVectorStore.from_texts(
    texts,
    embedding=emb
)

4. Setting up the retriever

In [36]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs = {"k":4},
)

In [37]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = OllamaLLM(model="gemma3",temperature=0 )

promtp = ChatPromptTemplate.from_template(
"""Răspundeți la întrebare doar pe baza următorului context.
Dacă contextul nu conține răspunsul, 
spuneți „Nu am suficiente informații pentru a răspunde la 
asta”. 
    Context: {context} 

    Întrebare: {question} 
    
    Răspuns:"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context":retriever | format_docs,"question":RunnablePassthrough()}
    | promtp
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("Ce este Teoria generală a sistemelor")
answer2 = rag_chain.invoke("What steps do industrial processes involve?")
print(answer)
print("-------")
print(answer2)


Teoria generală a sistemelor este o teorie despre sisteme desprinse de formele lor concrete. Extrage componentele esențiale și definitorii ale fiecărui tip de sistem, considerând complexitățile organizate. Se concentrează pe înțelegerea obiectelor care posedă proprietăți precum integralitatea, finalitatea, organizarea și autonomia relativă.
-------
Industrial processes encompass a wide range of steps, generally categorized into these key stages:

**1. Conception & Design:**

* **Market Research & Needs Assessment:** Identifying a market need and understanding customer requirements.
* **Conceptual Design:**  Brainstorming and outlining the basic idea of the product or process.
* **Detailed Design:**  Creating blueprints, schematics, and specifications for every component and stage of the process. This includes material selection, equipment sizing, and control system design.
* **Process Flow Diagrams (PFDs) & Piping & Instrumentation Diagrams (P&IDs):** Visual representations of the enti

In [41]:
print(answer2)

Nu am suficiente informații pentru a răspunde la asta.


In [38]:
results = rag_chain.invoke("Care este definita Teoriei sistemelor?")
results

'Conform textului, Teoria sistemelor este un concept din domeniul filozofiei, un model epistemologic interdisciplinar în care sistemele sunt utilizate pentru a descrie și a explica fenomene cu grad variabil de complexitate. Este o cunoaștere superioară, de nivel teoretic, a obiectelor care posedă proprietatea integralității, a mulțimilor compacte de entități care ființează ca unul. Mai precis, sistemele sunt grupuri de elemente interconectate care acționează conform unui set de reguli sau constrângeri pentru a forma un întreg unitar.\n'

In [39]:
results = rag_chain.invoke("Ce este Teoria generală a sistemelor")
results

'Teoria generală a sistemelor este o teorie despre sisteme desprinse de formele lor concrete. Extrage componentele esențiale și definitorii ale fiecărui tip de sistem, considerând complexitățile organizate. Se concentrează pe înțelegerea obiectelor care posedă proprietăți precum integralitatea, finalitatea, organizarea și autonomia relativă.'

Pattern 4: Evaluate Whater your pipeline Actually works

In [50]:
test_questions = [
    {
        "question":"ce seminficatie au mărimile u, x, y, v si z in teoria reglari automate?",
        "expected_keywords":["x","y","u"],
    },
    {
        "question": "What is task decomposition?",
        "expected_keywords": ["subgoal", "decompose", "smaller"],
    },
    {
        "question": "What are the types of agent memory?",
        "expected_keywords": ["short-term", "long-term", "sensory"],
    },
]

def evaluate_retrieval(retriever, test_cases):
    """Check if retrieved chunks contain expected keywords."""
    results = []
    for case in test_cases:
        docs = retriever.invoke(case["question"])
        retrieved_text = " ".join(d.page_content for d in docs).lower()

        found = [
            kw for kw in case["expected_keywords"]
            if kw.lower() in retrieved_text
        ]
        missing = [
            kw for kw in case["expected_keywords"]
            if kw.lower() not in retrieved_text
        ]

        score = len(found) / len(case["expected_keywords"])
        results.append({
            "question": case["question"],
            "score": score,
            "found": found,
            "missing": missing,
        })
        status = "PASS" if score >= 0.5 else "FAIL"
        print(f"[{status}] {case['question']} — {score:.0%}")
        if missing:
            print(f"  Missing: {missing}")

    avg = sum(r["score"] for r in results) / len(results)
    print(f"\nAverage retrieval score: {avg:.0%}")
    return results


evaluate_retrieval(retriever, test_questions)

for case in test_questions:
    print(rag_chain.invoke(case["question"]))
    print("-----------------------------------")
    

[PASS] ce seminficatie au mărimile u, x, y, v si z in teoria reglari automate? — 100%
[PASS] What is task decomposition? — 100%
[PASS] What are the types of agent memory? — 67%
  Missing: ['sensory']

Average retrieval score: 89%
În teoria reglării automate, variabilele u, x, y și z au următoarele semnificații:

*   **u (Input):** Este semnalul de intrare, controlul aplicat sistemului. Este semnalul care determină comportamentul sistemului.
*   **x (State):** Reprezintă starea internă a sistemului la un moment dat. Este o variabilă care descrie configurația internă a sistemului.
*   **y (Output):** Este variabila de ieșire, răspunsul sistemului la intrarea u. Este ceea ce măsurăm sau observăm din sistem.
*   **z (Measurement):** Este semnalul de măsurare al variabilei de ieșire y.

**În esență, un sistem de control este definit de o relație de tipul:**

y = f(x, u)

unde:

*   y este ieșirea sistemului
*   x este starea sistemului
*   u este intrarea (controlul)
*   f este o funcție ca

In [51]:
results = rag_chain.invoke("Care este definita Procesului industrial?")
results

'Răspunsul la întrebarea ta este complex și depinde de context. În general, procesul industrial se referă la **un set de activități și operațiuni care transformă intrările (materii prime, energie, capital) în ieșiri (produse finite, servicii) într-un mediu de producție.**\n\nIată o defalcare a elementelor cheie care definesc un proces industrial:\n\n*   **Transformare:** Procesul industrial implică modificarea materialelor, energiei sau informațiilor.\n*   **Intrări:**  Acestea includ resursele necesare pentru producție, cum ar fi materii prime, energie, apă, etc.\n*   **Ieșiri:** Acestea sunt produsele sau serviciile rezultate din procesul de producție.\n*   **Etape:** Procesul industrial este de obicei împărțit în mai multe etape, fiecare având un scop specific.\n*   **Control:** Procesul este monitorizat și controlat pentru a asigura calitatea și eficiența.\n*   **Mediu:** Procesul industrial operează într-un mediu specific, care poate include factori umani, sociali și de mediu.\n\n

In [53]:
results = rag_chain.invoke("Ce implica Procesul Fizic ?")
results

'Nu am suficiente informații pentru a răspunde la asta.'